# 🚲 Bike Sharing Demand: Empirical Regression Benchmark
**Authors:** Ujjwal Singh (262IS036) & Aryan (262IS007)
**Course:** CS801 Computing Lab, M.Tech, NITK Surathkal

### Objective:
Predict hourly bicycle rental count (`cnt`) across 17,379 records using pure classical (non-tree) regression algorithms:
1. Ordinary Least Squares (OLS) Linear Regression
2. Ridge Regression ($L_2$ Regularization)
3. Lasso Regression ($L_1$ Regularization)
4. ElasticNet Regression ($L_1 + L_2$)
5. Polynomial Regression (Degree 2 + Ridge)
6. K-Nearest Neighbors Regressor (KNN)
7. Support Vector Regressor (Linear SVR)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Load dataset
df = pd.read_csv("../../datasets/bike_sharing.csv")
print("Dataset Shape:", df.shape)
df.head()


## 1. Feature Selection & Leak-Free Train/Test Split
We drop ID columns (`instant`), raw date string (`dteday`), and component counts (`casual` and `registered`, which directly sum to `cnt`).
We partition 80% for training and 20% for testing before any scaling or encoding.


In [ ]:
drop_cols = ['instant', 'dteday', 'casual', 'registered']
df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns])

X = df_clean.drop(columns=['cnt'])
y = df_clean['cnt']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training samples:", len(X_train), "| Testing samples:", len(X_test))


## 2. Preprocessing with ColumnTransformer
- Numerical Features: `StandardScaler`
- Categorical Features: `OneHotEncoder(drop='first')` to prevent the dummy variable trap


In [ ]:
num_features = ['temp', 'atemp', 'hum', 'windspeed']
cat_features = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
    ]
)


## 3. Training & Benchmarking Models

In [ ]:
models = {
    'Linear Regression (OLS)': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1, random_state=42),
    'ElasticNet (alpha=0.1)': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    'Polynomial Ridge (d=2)': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('reg', Ridge(alpha=10.0))
    ]),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5, weights='distance'),
    'Linear SVR': LinearSVR(C=1.0, random_state=42, max_iter=3000)
}

results = []
for name, model in models.items():
    if 'Polynomial' in name:
        pipe = Pipeline([
            ('prep', preprocessor),
            ('poly', model.named_steps['poly']),
            ('reg', model.named_steps['reg'])
        ])
    else:
        pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    
    pipe.fit(X_train, y_train)
    train_r2 = r2_score(y_train, pipe.predict(X_train))
    test_preds = pipe.predict(X_test)
    test_r2 = r2_score(y_test, test_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    
    results.append({
        'Model': name,
        'Train R²': round(float(train_r2), 4),
        'Test R²': round(float(test_r2), 4),
        'Test MAE': round(float(test_mae), 2),
        'Test RMSE': round(float(test_rmse), 2)
    })

scorecard = pd.DataFrame(results).sort_values('Test R²', ascending=False)
scorecard
